# Tutorial 1: Quick Start

This tutorial shows the core workflow: load data, search for similar patterns, project forward, and visualize.

We use synthetic data so this notebook is self-contained — no external data files needed.

In [ ]:
import numpy as np
from the_similarity import load, search, project, plot, Config

## 1. Create Synthetic Price Data

We'll generate a random walk with embedded repeating patterns — a V-shaped dip that appears multiple times.

In [ ]:
np.random.seed(42)

# Base random walk
n = 2000
returns = np.random.normal(0.0002, 0.01, n)
prices = 100 * np.exp(np.cumsum(returns))

# Embed a V-shaped dip pattern at known positions
pattern_length = 60
dip = np.concatenate([
    np.linspace(0, -0.08, pattern_length // 2),
    np.linspace(-0.08, 0.02, pattern_length // 2),
])

# Plant the pattern at positions 300, 800, 1300
for start in [300, 800, 1300]:
    for i, d in enumerate(dip):
        prices[start + i] *= (1 + d)

print(f"Generated {len(prices)} bars of synthetic price data")
print(f"Price range: {prices.min():.2f} - {prices.max():.2f}")

## 2. Load Data

`load()` accepts numpy arrays, DataFrames, CSVs, or parquet files.

In [ ]:
history = load(prices)
print(f"Loaded TimeSeries: {len(history)} bars")

# Use the last planted pattern as our query
query = load(prices[1300:1360])
print(f"Query: {len(query)} bars")

## 3. Search for Similar Patterns

The search pipeline runs:
1. SAX + MASS prefilter (eliminates ~95% of candidates)
2. DTW + Pearson scoring on survivors
3. Tier 2 enrichment (7 methods) on top 20
4. Composite confidence score (0-100)

In [ ]:
# Use stride=3 for speed (checks every 3rd position)
results = search(query, history, top_k=5, stride=3)
results.summary()

## 4. Inspect Matches

Each match has a full score breakdown across all 9 methods.

In [ ]:
best = results.best
print(f"Best match: bars {best.start_idx}-{best.end_idx}")
print(f"Confidence: {best.confidence_score:.1f}/100")
print(f"Regime: {best.regime}")
print()
print("Score breakdown:")
b = best.score_breakdown
for field in ['dtw', 'pearson_warped', 'bempedelis_r2', 'koopman',
              'wavelet_spectrum', 'emd', 'tda', 'transfer_entropy']:
    print(f"  {field}: {getattr(b, field):.3f}")

## 5. Project Forward

Generate a probabilistic forecast cone from the matched patterns.

In [ ]:
forecast = project(results, history, forward_bars=30)

print(f"Forecast: {forecast.bars} bars forward")
print(f"Percentiles: {forecast.percentiles}")
print(f"Paths used: {forecast.all_paths.shape[0]}")
print()
print(f"P50 endpoint: {forecast.curves[50][-1]:+.4f} ({forecast.curves[50][-1]*100:+.2f}%)")
print(f"P10 endpoint: {forecast.curves[10][-1]:+.4f} (bearish)")
print(f"P90 endpoint: {forecast.curves[90][-1]:+.4f} (bullish)")

## 6. Visualize

In [ ]:
plot(results, forecast, top_n=3, show=True)

## Next Steps

- **Tutorial 2**: Configuration — tuning methods, weights, and performance
- **Tutorial 3**: Backtesting — validating the pipeline on historical data
- See `docs/API_REFERENCE.md` for the full API surface